In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/vinuit/simclr200/pytorch/default/1/checkpoint_epoch_200.pth


In [2]:
### Install necessary library
!pip install lightly

### Import essential modules
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms

import lightly.data as data
import lightly.models as lightly_models
import lightly.loss as loss

print('Libraries successfully imported.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.6/863.6 kB 16.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.8 MB/s eta 0:00:00
Libraries successfully imported.


In [3]:
import torch
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import lightly.data as data

# 1. Define image size for CIFAR-10
input_size = 32

# 2. Use SimCLRCollateFunction for the augmentation pipeline
collate_fn = data.SimCLRCollateFunction(
    input_size=input_size,
    gaussian_blur=0.7,
)

# 3. Download the CIFAR-10 training set
base_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True
)

# Wrap with LightlyDataset to ensure compatibility with SimCLRCollateFunction
# This provides the (image, label, filename) format expected by the collate function
train_dataset = data.LightlyDataset.from_torch_dataset(base_dataset)

# 4. Create a PyTorch DataLoader
batch_size = 1024
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True,
    num_workers=2,
    persistent_workers=True
)

print(f'DataLoader initialized with batch size {batch_size}.')
print(f'Total training samples: {len(train_dataset)}.')

100%|██████████| 170M/170M [00:02<00:00, 78.8MB/s] 


DataLoader initialized with batch size 1024.
Total training samples: 50000.


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from lightly.models.modules import SimCLRProjectionHead
from lightly.loss import NTXentLoss

# 1. Define the backbone using ResNet-18
resnet = models.resnet18(weights=None)
backbone = nn.Sequential(*list(resnet.children())[:-1])
input_dim = 512 # ResNet-18 last layer before FC

# 2. Create the SimCLR model with a projection head
class SimCLR(nn.Module):
    def __init__(self, backbone, input_dim, hidden_dim=512, out_dim=128):
        super(SimCLR, self).__init__()
        self.backbone = backbone
        self.projection_head = SimCLRProjectionHead(input_dim, hidden_dim, out_dim)

    def forward(self, x):
        x = self.backbone(x).flatten(start_dim=1)
        z = self.projection_head(x)
        z = F.normalize(z, dim=1)   # L2 normalize
        return z

model = SimCLR(backbone, input_dim)

# 3. Initialize NT-Xent loss
criterion = NTXentLoss(temperature=0.5)

# 4. Define Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

# 5. Move to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model.to(device)
criterion.to(device)

print(f'Model initialized and moved to {device}.')
print(f'Loss function: NTXentLoss, Optimizer: Adam.')

Using 2 GPUs
Model initialized and moved to cuda.
Loss function: NTXentLoss, Optimizer: Adam.


In [5]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def knn_evaluate(model, train_loader, device, k=10):
    model.eval()

    # Build feature bank for ALL points
    feature_bank = []
    label_bank = []
    for batch in train_loader:
        (x0, _), y, _ = batch
        x0 = x0.to(device)
        y = y.to(device)
        features = model(x0)
        features = F.normalize(features, dim=1)
        feature_bank.append(features)
        label_bank.append(y)

    feature_bank = torch.cat(feature_bank, dim=0)  # [N, D]
    label_bank = torch.cat(label_bank, dim=0)      # [N]

    correct = 0
    total = 0
    for batch in train_loader:
        (x0, _), y, _ = batch
        x0 = x0.to(device)
        y = y.to(device)
        features = model(x0)
        features = F.normalize(features, dim=1)

        # Row by row to save memory
        pred_labels = []
        for i in range(features.size(0)):
            sim_row = torch.mv(feature_bank, features[i])  # [N]
            top_indices = sim_row.topk(k=k+1).indices[1:]  # skip self
            neighbor_labels = label_bank[top_indices]
            counts = torch.bincount(neighbor_labels, minlength=10)
            pred_labels.append(counts.argmax())

        pred_labels = torch.stack(pred_labels)
        correct += (pred_labels == y).sum().item()
        total += y.size(0)

    acc = 100.0 * correct / total
    return acc

In [6]:
import os
import torch

def train_simclr(
    model,
    train_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    start_epoch=0,
    epochs=500,
    patience=20,
    save_dir="/kaggle/working",
    save_freq=20,
    knn_freq=8
):
    os.makedirs(save_dir, exist_ok=True)

    best_loss = float('inf')
    counter = 0
    epoch_losses = []
    epsilon = 0.001

    print(f'Starting training from epoch {start_epoch+1} on {device}...')

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            (x0, x1), _, _ = batch
            x0, x1 = x0.to(device), x1.to(device)

            z0 = model(x0)
            z1 = model(x1)
            loss = criterion(z0, z1)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        epoch_losses.append(avg_loss)

        # Step scheduler (IMPORTANT)
        if scheduler is not None:
            scheduler.step()

        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

        if (epoch + 1) % knn_freq == 0:
            knn_acc = knn_evaluate(model, train_loader, device)
            print(f'kNN Accuracy: {knn_acc:.2f}%')

        # Save best model
        if avg_loss < best_loss - epsilon:
            best_loss = avg_loss
            counter = 0

            best_path = os.path.join(save_dir, "simclr_best.pth")
            torch.save({
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict() if scheduler else None,
                'loss': best_loss
            }, best_path)

        else:
            counter += 1

        # Periodic checkpoint
        if (epoch + 1) % save_freq == 0:
            ckpt_path = os.path.join(save_dir, f"checkpoint_epoch_{epoch+1}.pth")
            torch.save({
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict() if scheduler else None,
                'loss': avg_loss
            }, ckpt_path)
            print(f"Checkpoint saved: {ckpt_path}")

        # Early stopping (less aggressive now)
        if counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

    print('Training completed.')
    return epoch_losses

In [7]:
from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=500,   # full cosine cycle over all epochs
    eta_min=0       # final LR
)
#train_simclr(model,train_loader,criterion,optimizer,scheduler,device)

In [8]:
ckpt = torch.load("/kaggle/input/models/vinuit/simclr200/pytorch/default/1/checkpoint_epoch_200.pth")

model.load_state_dict(ckpt['model'])
optimizer.load_state_dict(ckpt['optimizer'])


train_simclr(model,train_loader,criterion,optimizer,scheduler,device,start_epoch=200)

Starting training from epoch 201 on cuda...
Epoch [201/500], Loss: 6.2536
Epoch [202/500], Loss: 6.2533
Epoch [203/500], Loss: 6.2496
Epoch [204/500], Loss: 6.2513
Epoch [205/500], Loss: 6.2478
Epoch [206/500], Loss: 6.2495
Epoch [207/500], Loss: 6.2511
Epoch [208/500], Loss: 6.2503
kNN Accuracy: 61.26%
Epoch [209/500], Loss: 6.2516
Epoch [210/500], Loss: 6.2502
Epoch [211/500], Loss: 6.2493
Epoch [212/500], Loss: 6.2451
Epoch [213/500], Loss: 6.2459
Epoch [214/500], Loss: 6.2492
Epoch [215/500], Loss: 6.2446
Epoch [216/500], Loss: 6.2467
kNN Accuracy: 61.47%
Epoch [217/500], Loss: 6.2476
Epoch [218/500], Loss: 6.2434
Epoch [219/500], Loss: 6.2416
Epoch [220/500], Loss: 6.2491
Checkpoint saved: /kaggle/working/checkpoint_epoch_220.pth
Epoch [221/500], Loss: 6.2451
Epoch [222/500], Loss: 6.2446
Epoch [223/500], Loss: 6.2457
Epoch [224/500], Loss: 6.2433
kNN Accuracy: 61.37%
Epoch [225/500], Loss: 6.2410
Epoch [226/500], Loss: 6.2426
Epoch [227/500], Loss: 6.2432
Epoch [228/500], Loss: 6

[6.25363352894783,
 6.253308594226837,
 6.249649653832118,
 6.251271744569142,
 6.2477822204430895,
 6.24952815969785,
 6.251140813032786,
 6.250268518924713,
 6.25160809357961,
 6.250238170226415,
 6.249317179123561,
 6.245115886131923,
 6.245920618375142,
 6.249220252037048,
 6.2445882360140486,
 6.246743510166804,
 6.247574915488561,
 6.243389924367269,
 6.241602172454198,
 6.249055912097295,
 6.245055586099625,
 6.244610865910848,
 6.245709369579951,
 6.243254552284877,
 6.241005957126617,
 6.242583900690079,
 6.243211974700292,
 6.246658106644948,
 6.241977135340373,
 6.243943641583125,
 6.247591892878215,
 6.238719612360001,
 6.241412719090779,
 6.242406199375789,
 6.245419840017955,
 6.2402883768081665,
 6.243771860996882,
 6.2421534061431885,
 6.239266822735469,
 6.244005272785823,
 6.239298899968465,
 6.242529521385829,
 6.241131156682968,
 6.239669511715571,
 6.232442865769069,
 6.2374076048533125,
 6.2369133333365125,
 6.240054776271184,
 6.23580797513326,
 6.237425178289413